# Package Imoport and Installation

In [1]:
%%capture
!pip install ultralytics
!pip install onnx
!pip install onnx_tf
!pip install matplotlib
!pip install protobuf==3.20.1
!pip install keras==2.11.0
!pip install tensorflow-estimator==2.11.0
!pip install tensorboard==2.11.0
!pip install torchinfo
!pip install torchview

In [2]:
%%capture
import os
import json, datetime
import torch
from ultralytics import YOLO
from torchinfo import summary
from yolov8_EE_network_backbone import YOLOv8n_EE #implemented network leveraging ultralytics and torch utils
from torch_graph import model_graph #implemented graph service for demonstrating model's architecture

In [3]:
#dataset variables
ROOT_DIR= '/root/Desktop/workspace/datasets/obstacle-detector/dataset' #our dataset's directory
test_folder= os.path.join(ROOT_DIR, 'test/images/')
DATA_YAML= os.path.join(ROOT_DIR, 'data.yaml') #dataset yaml file

MODEL_YAML = "/root/Desktop/workspace/yolov8n_ee.yaml"
DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu' #device declearation for model traning
IMG_SIZE= 640 #size of image and input tensors
BATCH_SIZE= 16
EPOCHS= 80
class_count = 18

(Optional) **T4-GPU Availability Checking and Drive Mounting for Providing Dataset Access**

If you are running this code on a local computer or server you may pass this section.

In [6]:
from google.colab import drive
drive.mount('/content/drive/')

%cd /content/drive/MyDrive/EE using ultralitycs
!ls

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/MyDrive/EE using ultralitycs
'Copy of Copy of YOLOv8_EarlyExit.ipynb'   torch_graph.py
 __pycache__				   yolo11n.pt
 requirements.txt			   yolo_early_exit_torchview.png
 runs					   YOLOv8_EarlyExit.ipynb
 tensor.pt				   yolov8_EE_network.py
 test.ipynb				   yolov8n_ee.yaml
 torch_graph.ipynb			   yolov8n.pt


In [4]:
print("Torch Cuda - GPU informaiton:")
print("availablity: ", torch.cuda.is_available())
print("name: ", torch.cuda.get_device_name())
print("properties: ", torch.cuda.get_device_properties(device= 0))

Torch Cuda - GPU informaiton:
availablity:  True
name:  NVIDIA GeForce RTX 3090
properties:  _CudaDeviceProperties(name='NVIDIA GeForce RTX 3090', major=8, minor=6, total_memory=24135MB, multi_processor_count=82)


# Early Exit Implementation in YOLOv8n

## Adding Early-Exit layers using ultralytics utilities

Here we will reimplement YOLOv8's architecture and will add 5 detect layers in backbone and neck aim to increase its infrence time and decrease its computational overheads while we are tending to maintaion accuracy's rates static as far as its possible.

In [5]:
#Build model and extract its summary
yolo_ee = YOLO(MODEL_YAML)

yolo_ee.model = YOLOv8n_EE(nc=class_count,model_yaml=MODEL_YAML).to(DEVICE)

summary(yolo_ee)

/opt/conda/lib/python3.8/site-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /pytorch/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


Layer (type:depth-idx)                             Param #
YOLO                                               --
├─YOLOv8n_EE: 1-1                                  --
│    └─Conv: 2-1                                   --
│    │    └─Conv2d: 3-1                            1,728
│    │    └─BatchNorm2d: 3-2                       128
│    │    └─SiLU: 3-3                              --
│    └─Conv: 2-2                                   --
│    │    └─Conv2d: 3-4                            73,728
│    │    └─BatchNorm2d: 3-5                       256
│    │    └─SiLU: 3-6                              --
│    └─C2f: 2-3                                    --
│    │    └─Conv: 3-7                              33,280
│    │    └─Conv: 3-8                              82,176
│    │    └─ModuleList: 3-9                        886,272
│    └─Detect: 2-4                                 --
│    │    └─ModuleList: 3-10                       115,008
│    │    └─ModuleList: 3-11                      

In [ ]:
model_graph(
    yolo_ee,
    expand_nested=False,
    graph_name="YOLO_EE_Graph",
)

In [6]:
results = yolo_ee.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
    name='yolov8n-ee',
    pretrained=False
)


Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (8)

train: Scanning /workspace/datasets/obstacle-detector/dataset/train/labels.cache... 13937 images, 83 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 13938/13938 7.9Mit/s 0.0s
val: Scanning /workspace/datasets/obstacle-detector/dataset/valid/labels.cache... 2717 images, 11 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2717/2717 859.2Kit/s 0.0s


[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warn

       1/80      3.45G      3.189      4.762      4.058          6        640: 100% ━━━━━━━━━━━━ 872/872 7.0it/s 2:05<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 6.3it/s 13.6s0.2s
       2/80      3.67G      2.272      3.922      2.856          3        640: 100% ━━━━━━━━━━━━ 872/872 7.5it/s 1:56<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 7.7it/s 11.0s0.1s
       3/80      3.67G      1.747      3.062       2.24          4        640: 100% ━━━━━━━━━━━━ 872/872 8.0it/s 1:49<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 7.2it/s 11.8s0.1s
       4/80      3.67G      1.581      2.541      2.015          1        640: 100% ━━━━━━━━━━━━ 872/872 7.5it/s 1:56<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8

[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)


      71/80      3.97G     0.8163     0.6576      1.303          1        640: 100% ━━━━━━━━━━━━ 872/872 8.5it/s 1:42<0.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 8.0it/s 10.7s0.1s
      72/80      3.97G     0.7951      0.633      1.286          1        640: 100% ━━━━━━━━━━━━ 872/872 8.6it/s 1:42<0.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 7.9it/s 10.8s0.1s
      73/80      3.97G      0.781     0.6171      1.275          2        640: 100% ━━━━━━━━━━━━ 872/872 8.3it/s 1:44<0.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 85/85 7.7it/s 11.0s0.1s
      74/80      3.97G     0.7785     0.6155      1.277          1        640: 100% ━━━━━━━━━━━━ 872/872 8.5it/s 1:43<0.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━


## Test and Validation

In [7]:
model_path = "/root/Desktop/workspace/runs/detect/yolov8n-ee2/weights/best.pt"
# 1) Load the Ultralytics model from the .pt
model = YOLO(model_path)

test_results = model.predict(source=test_folder, save=True, show=False)
print("Prediction complete. Check the 'runs/detect/predict/' folder for the saved results with bounding boxes.")

Prediction complete. Check the 'runs/detect/predict/' folder for the saved results with bounding boxes.


In [8]:
# Test and Valifation
model.val(data=DATA_YAML,split="test")

print("Validation complete. Check the 'runs/detect/valid/' folder for the saved validation statics.")

val: Scanning /workspace/datasets/obstacle-detector/dataset/test/labels.cache... 1157 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1157/1157 735.7Kit/s 0.0s


[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 5.4it/s 13.4s<0.1s
Validation complete. Check the 'runs/detect/valid/' folder for the saved validation statics.
